# Remote Submission of MRChem on Fram


In [7]:
from remotemanager.connection.computers.base import BaseComputer, \
    required, optional

def fram_parser(resources):
    from remotemanager.connection.computers.base import format_time
    output = []
    
    cores_per_node = 32
    if resources['mpi_per_node']:
        if not resources['nodes']:
            raise RuntimeError('nodes must be specified with mpi_per_node')
        if resources['mpi_per_node'].value > cores_per_node:
            raise ValueError(f'mpi_per_node cannot be greater than cores_per_node ({cores_per_node})')

        nodes = resources['nodes'].value
        ntasks = resources['mpi_per_node'].value * nodes
        omp = resources["omp"].value
        
        output.append(f"#SBATCH --nodes={nodes}")
        output.append(f"#SBATCH --ntasks={ntasks}")
    else:
        # extract our mpi and omp request, calculate total cores
        mpi = resources["mpi"].value
        omp = resources["omp"].value
        ntasks = mpi
        ncores = mpi * omp
        print(f'{ncores} total cores requested')
        
        nodes = int(round(max(1, ncores / cores_per_node)))
        output.append(f"#SBATCH --nodes={nodes}")
        output.append(f"#SBATCH --ntasks={ntasks}")

    #if not resources["partition"]:
    #   resources["partition"] = resources["queue"].value

    #options['--export'] = 'none'    
    for k, v in resources.items():
        if k in ['nodes', 'mpi' 'mpi_per_node']:
            continue
        if k == "time":
            output.append(f"#SBATCH --{v.flag}={format_time(v.value)}")
        elif v:
            output.append(f"#SBATCH --{v.flag}={v.value}")
    
    output.append(f'\nexport OMP_NUM_THREADS={omp}')

    return output


class Fram(BaseComputer):
    """
    class for connecting to Fram
    """

    def __init__(self,
                 passfile: str,
                 module_purge=False,
                 **kwargs):

        kwargs['passfile'] = passfile

        if 'host' not in kwargs:
            kwargs['host'] = 'fram.sigma2.no'

        super().__init__(**kwargs)

        self.submitter = 'sbatch'
        self.shebang = '#!/bin/bash'

        self.mpi = required('ntasks')
        self.mpi_per_node = required('tasks-per-node')
        self.omp = required('cpus-per-task')
        self.walltime = required('time')
        #self.memory = required('mem-per-cpu')        
        self.nodes = optional('nodes')
        self.jobname = optional('job-name')
        self.outfile = optional('output')
        self.errfile = optional('error')
        self.account = optional('account')
        
        #self.queue = optional('')
        #self.partition = optional('partition', 'normal')

        self.choice_groupings = [["mpi", "mpi_per_node"]]

        self._parser = fram_parser

In [13]:
from remotemanager import Dataset, URL

url = Fram(user='lra040')

url.mpi = 1
url.omp = 32
url.memory = '2GB'
url.walltime = "01:00:00"
url.account = "nn4654k"
url.extra = """export MRCHEM_MPIRUN='srun --distribution=cyclic:cyclic'
export MRCHEM_ROOT=/cluster/home/lra040/PROJECT_WORK/mrchem/install-1.0.0/bin
module --quiet purge
module load Python/3.7.4-GCCcore-8.3.0
module load intel/2019b
module load Eigen/3.3.7
module load SciPy-bundle/2020.03-intel-2020a-Python-3.8.2
export PYTHONPATH=/cluster/projects/nn4654k/lra040/bigdft-suite/jhbuild/sitecustomize:/cluster/projects/nn4654k/lra040/bigdft-suite/Build/install/lib/python3.6/site-packages:$PYTHONPATH
"""
#source /cluster/home/lra040/PROJECT_WORK/bigdft-suite/Build/install/bin/bigdftvars.sh
url.python = "python3"

remote_dir = f'/cluster/home/lra040/PROJECT_WORK/core_mrchem'
local_dir = 'testing'

Double check connection by checking the current directory

In [14]:
url.cmd('pwd')

/cluster/home/lra040

Define a function to run MRChem

In [15]:
def run_mrchem(sys):
    from BigDFT.Interop.MRChemInterop import MRChemCalculator
        
    Ha2eV = 27.211396132
    
    # define an MRChem input dictionary which can be converted to JSON format
    inp = {}
    inp["WaveFunction"] = {"method": "PBE"}
    inp["world_prec"] = 1.0e-3
    
    # define a calculator to run MRChem  
    code = MRChemCalculator(skip=True)
    log = code.run(sys=sys, input=inp, name="test")

    # extract some properties BigDFT-style
    energy = log.energy
    #print(log["output"]["properties"]["scf_energy"])
    
    return energy

In [16]:
from BigDFT.Database.Molecules import get_molecule
sys = get_molecule("H2O")

In [17]:
from remotemanager.serialisation import serialdill
from os.path import join

mrchem_runs = Dataset(function=run_mrchem, url=url, name='mrchem_test',
                     remote_dir=join(remote_dir, local_dir),
                     local_dir=local_dir, serialiser=serialdill(), skip=False)

run_args = {} 
run_args['mpi'] = 1
run_args['omp'] = 32
run_args['time'] = 6 * 3600

mrchem_run_names = []
run_name = 'mrchem_test'
run_args['jobname'] = run_name
    
func_kwargs = dict(sys=sys)
mrchem_runs.append_run(args=func_kwargs, local_dir=local_dir, **run_args)
mrchem_run_names.append(run_name)    

mrchem_runs.run()

appended run runner-0
assessing run for runner mrchem_test-4042368c-runner-0... checks passed, running
32 total cores requested


In [18]:
print(mrchem_runs.is_finished)

checking remotely for finished runs
[True]


In [20]:
mrchem_runs.fetch_results()

checking remotely for finished runs


In [21]:
print(mrchem_runs.results)

[-76.38528902085758]
